In [1]:

import os, glob
import numpy as np
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import GBTRegressor
from pyspark.storagelevel import StorageLevel

# ---------------- Spark (restart optionally) ----------------
try:
    spark.stop()
except:
    pass

spark = (SparkSession.builder
         .appName("favorita-multi-horizon-no-oom")
         .config("spark.sql.shuffle.partitions", "24")
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

BASE = "/kaggle/input"
def find_file(filename: str) -> str:
    for d in glob.glob(os.path.join(BASE, "*")):
        cand = os.path.join(d, filename)
        if os.path.exists(cand):
            return cand
    for root, _, files in os.walk(BASE):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not find {filename} under {BASE}")

train_path  = find_file("train.csv")
items_path  = find_file("items.csv")
stores_path = find_file("stores.csv")
trans_path  = find_file("transactions.csv")
oil_path    = find_file("oil.csv")
hol_path    = find_file("holidays_events.csv")

print("Paths:")
print("train:", train_path)
print("items:", items_path)
print("stores:", stores_path)
print("transactions:", trans_path)
print("oil:", oil_path)
print("holidays:", hol_path)

# ---------------- CONFIG ----------------
START_DATE = "2015-06-01"
END_DATE   = "2017-08-16"

HORIZON_MAX = 30
HORIZONS = list(range(1, HORIZON_MAX + 1))

BASE_STRIDE_DAYS = 7     # <<< КЛЮЧ: беремо base_date кожні 7 днів (менше даних)

K_FAMILIES = 3
K_STORES   = 1
TOP_ITEMS_PER_FAMILY = 80

TRAIN_END = "2017-06-01"
VAL_END   = "2017-08-01"

LAGS = [1, 7, 14, 28]
ROLL_WINDOWS = [7, 14, 28]

# ---------------- Schemas ----------------
train_schema = StructType([
    StructField("id", LongType(), True),
    StructField("date", StringType(), True),
    StructField("store_nbr", IntegerType(), True),
    StructField("item_nbr", IntegerType(), True),
    StructField("unit_sales", DoubleType(), True),
    StructField("onpromotion", StringType(), True),
])

items_schema = StructType([
    StructField("item_nbr", IntegerType(), True),
    StructField("family", StringType(), True),
    StructField("class", IntegerType(), True),
    StructField("perishable", IntegerType(), True),
])

stores_schema = StructType([
    StructField("store_nbr", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("type", StringType(), True),
    StructField("cluster", IntegerType(), True),
])

trans_schema = StructType([
    StructField("date", StringType(), True),
    StructField("store_nbr", IntegerType(), True),
    StructField("transactions", IntegerType(), True),
])

oil_schema = StructType([
    StructField("date", StringType(), True),
    StructField("dcoilwtico", DoubleType(), True),
])

hol_schema = StructType([
    StructField("date", StringType(), True),
    StructField("type", StringType(), True),
    StructField("locale", StringType(), True),
    StructField("locale_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("transferred", BooleanType(), True),
])

# ==========================================================
# 1) Select subset
# ==========================================================
print("\n--- Phase 1: Selecting subset ---")

items_full  = spark.read.option("header","true").schema(items_schema).csv(items_path)
stores_full = spark.read.option("header","true").schema(stores_schema).csv(stores_path)

train_min = (spark.read.option("header","true").schema(train_schema).csv(train_path)
             .filter(F.col("date") >= START_DATE)
             .select("store_nbr","item_nbr"))

train_with_family = train_min.join(F.broadcast(items_full.select("item_nbr","family")), "item_nbr", "inner")

top_fam = (train_with_family.groupBy("family").count()
           .orderBy(F.desc("count")).limit(K_FAMILIES).collect())
target_families = [r["family"] for r in top_fam]
print("Selected families:", target_families)

train_fam = train_with_family.filter(F.col("family").isin(target_families))

top_store = (train_fam.groupBy("store_nbr").count()
             .orderBy(F.desc("count")).limit(K_STORES).collect())
target_stores = [int(r["store_nbr"]) for r in top_store]
print("Selected stores:", target_stores)

from pyspark.sql.window import Window as W
w_items = W.partitionBy("family").orderBy(F.desc("count"))
item_counts = (train_fam.filter(F.col("store_nbr").isin(target_stores))
               .groupBy("family","item_nbr").count())

top_items_df = (item_counts.withColumn("rn", F.row_number().over(w_items))
                .filter(F.col("rn") <= TOP_ITEMS_PER_FAMILY)
                .select("item_nbr").distinct())

target_items = [int(r["item_nbr"]) for r in top_items_df.collect()]
print(f"Selected items total: {len(target_items)}")

del train_min, train_with_family, train_fam, item_counts

# ==========================================================
# 2) Load filtered data
# ==========================================================
print("\n--- Phase 2: Loading filtered tables ---")

stores = (stores_full.filter(F.col("store_nbr").isin(target_stores))
          .select("store_nbr","city","state",F.col("type").alias("store_type"),"cluster"))

items = (items_full.filter(F.col("family").isin(target_families))
         .filter(F.col("item_nbr").isin(target_items))
         .select("item_nbr","family","class","perishable"))

train_raw = (spark.read.option("header","true").schema(train_schema).csv(train_path)
             .withColumn("date", F.to_date("date"))
             .filter(F.col("date").between(F.to_date(F.lit(START_DATE)), F.to_date(F.lit(END_DATE))))
             .filter(F.col("store_nbr").isin(target_stores)))

train_raw = train_raw.join(F.broadcast(items.select("item_nbr")), "item_nbr", "left_semi")
print("Train filtered rows:", train_raw.count())

hol = (spark.read.option("header","true").schema(hol_schema).csv(hol_path)
       .withColumn("date", F.to_date("date"))
       .filter(F.col("date").between(F.to_date(F.lit(START_DATE)), F.to_date(F.lit(END_DATE)))))
if "locale" not in hol.columns and "lacale" in hol.columns:
    hol = hol.withColumnRenamed("lacale", "locale")

# ==========================================================

print("\n--- Phase 3: Building dense grid + lags ---")

sales_actual = (train_raw
    .withColumn("onpromotion_bool",
                F.when(F.lower(F.col("onpromotion")) == "true", F.lit(1))
                 .when(F.lower(F.col("onpromotion")) == "false", F.lit(0))
                 .otherwise(F.lit(0)))
    .withColumn("unit_sales_clipped", F.when(F.col("unit_sales") < 0, F.lit(0.0)).otherwise(F.col("unit_sales")))
    .withColumn("label_today", F.log1p(F.col("unit_sales_clipped")))
    .select("date","store_nbr","item_nbr","label_today","onpromotion_bool")
)

date_dim = spark.sql(
    f"SELECT explode(sequence(to_date('{START_DATE}'), to_date('{END_DATE}'), interval 1 day)) as date"
)

# calendar features (known in future)
store_date_base = (date_dim.crossJoin(F.broadcast(stores.select("store_nbr")))
    .withColumn("dow", F.dayofweek("date"))
    .withColumn("dom", F.dayofmonth("date"))
    .withColumn("weekofyear", F.weekofyear("date"))
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .withColumn("is_weekend", (F.col("dow").isin([1,7])).cast("int"))
)

# holidays (known in future)
hol_clean = hol.filter(F.col("transferred") == F.lit(False)).drop("transferred")

hol_nat = hol_clean.filter(F.col("locale") == "National").crossJoin(F.broadcast(stores.select("store_nbr")))
hol_reg = (hol_clean.filter(F.col("locale") == "Regional")
           .join(F.broadcast(stores.select("store_nbr","state")),
                 stores.state == hol_clean.locale_name, "inner"))
hol_loc = (hol_clean.filter(F.col("locale") == "Local")
           .join(F.broadcast(stores.select("store_nbr","city")),
                 stores.city == hol_clean.locale_name, "inner"))

hol_store = (hol_nat.select("date","store_nbr","type")
             .unionByName(hol_reg.select("date","store_nbr","type"))
             .unionByName(hol_loc.select("date","store_nbr","type")))

holiday_features = (hol_store.groupBy("date","store_nbr").agg(
    F.count("*").alias("holiday_count"),
    F.max((F.col("type") == "Holiday").cast("int")).alias("is_holiday"),
    F.max((F.col("type") == "Event").cast("int")).alias("is_event"),
    F.max((F.col("type") == "Additional").cast("int")).alias("is_additional"),
    F.max((F.col("type") == "Bridge").cast("int")).alias("is_bridge"),
    F.max((F.col("type") == "Work Day").cast("int")).alias("is_workday"),
    F.max((F.col("type") == "Transfer").cast("int")).alias("is_transfer"),
))

store_date_base = (store_date_base
    .join(holiday_features, ["date","store_nbr"], "left")
    .fillna({"holiday_count":0,"is_holiday":0,"is_event":0,"is_additional":0,
             "is_bridge":0,"is_workday":0,"is_transfer":0})
)

future_cols = ["dow","dom","weekofyear","month","year","is_weekend",
               "holiday_count","is_holiday","is_event","is_additional","is_bridge","is_workday","is_transfer"]

store_date_t = store_date_base.select(
    F.col("date").alias("target_date"),
    "store_nbr",
    *[F.col(c).alias(f"{c}_t") for c in future_cols]
)

# dense grid store-item-date
store_item = sales_actual.select("store_nbr","item_nbr").distinct()

base_grid = (date_dim.crossJoin(store_item)
    .join(sales_actual, ["date","store_nbr","item_nbr"], "left")
    .fillna({"label_today":0.0,"onpromotion_bool":0})
    .join(store_date_base, ["date","store_nbr"], "left")
    .join(F.broadcast(stores), ["store_nbr"], "left")
    .join(F.broadcast(items), ["item_nbr"], "left")
    .fillna({"family":"UNKNOWN","city":"UNKNOWN","state":"UNKNOWN","store_type":"UNKNOWN"})
    .fillna({"class":0,"perishable":0,"cluster":0})
)

# lags/rolls
w_si = Window.partitionBy("store_nbr","item_nbr").orderBy("date")
for lag in LAGS:
    base_grid = base_grid.withColumn(f"lag{lag}_label", F.lag("label_today", lag).over(w_si))
for rw in ROLL_WINDOWS:
    base_grid = base_grid.withColumn(f"roll{rw}_mean_label", F.avg("label_today").over(w_si.rowsBetween(-rw, -1)))

base_grid = base_grid.fillna({f"lag{lag}_label":0.0 for lag in LAGS})
base_grid = base_grid.fillna({f"roll{rw}_mean_label":0.0 for rw in ROLL_WINDOWS})

# ==========================================================
# 4) MULTI-HORIZON but with base_date stride
# ==========================================================
h_array = F.array(*[F.lit(h) for h in HORIZONS])

# <<< КЛЮЧ: беремо base_date не кожен день, а кожні BASE_STRIDE_DAYS >>>
base_grid_strided = (base_grid
    .withColumn("dd", F.datediff("date", F.to_date(F.lit(START_DATE))))
    .filter(F.pmod(F.col("dd"), F.lit(BASE_STRIDE_DAYS)) == 0)
    .drop("dd")
)

df_base = (base_grid_strided
    .withColumn("horizon", F.explode(h_array))
    .withColumn("target_date", F.date_add(F.col("date"), F.col("horizon")))
    .filter(F.col("target_date") <= F.to_date(F.lit(END_DATE)))
)

target_lookup = base_grid.select(
    F.col("date").alias("target_date"),
    "store_nbr","item_nbr",
    F.col("label_today").alias("label_target"),
    F.col("onpromotion_bool").alias("onpromotion_target")
)

df = (df_base
    .join(target_lookup, ["target_date","store_nbr","item_nbr"], "left")
    .fillna({"label_target":0.0,"onpromotion_target":0})
    .join(store_date_t, ["target_date","store_nbr"], "left")
    .fillna({f"{c}_t":0 for c in future_cols})
    .withColumn("store_id", F.col("store_nbr").cast("string"))
    .withColumn("item_id",  F.col("item_nbr").cast("string"))
)

# ==========================================================
# 5) Split by target_date
# ==========================================================
train_df = df.filter(F.col("target_date") <  F.to_date(F.lit(TRAIN_END)))
val_df   = df.filter((F.col("target_date") >= F.to_date(F.lit(TRAIN_END))) & (F.col("target_date") < F.to_date(F.lit(VAL_END))))
test_df  = df.filter(F.col("target_date") >= F.to_date(F.lit(VAL_END)))

print("\nSplits by target_date (STRIDED):")
print("train rows:", train_df.count())
print("val rows:",   val_df.count())
print("test rows:",  test_df.count())

# ==========================================================
# 6) Features pipeline (NO OHE)
# ==========================================================
label_col = "label_target"

cat_cols = ["family","store_type","city","state","store_id","item_id"]
num_cols = [
    "horizon",
    "class","perishable","cluster",
    "onpromotion_target",
    *[f"lag{lag}_label" for lag in LAGS],
    *[f"roll{rw}_mean_label" for rw in ROLL_WINDOWS],
    *[f"{c}_t" for c in future_cols],
]

df = df.fillna({c:0 for c in num_cols}).fillna({c:"UNKNOWN" for c in cat_cols})
train_df = train_df.fillna({c:0 for c in num_cols}).fillna({c:"UNKNOWN" for c in cat_cols})
val_df   = val_df.fillna({c:0 for c in num_cols}).fillna({c:"UNKNOWN" for c in cat_cols})
test_df  = test_df.fillna({c:0 for c in num_cols}).fillna({c:"UNKNOWN" for c in cat_cols})

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]

assembled_input = num_cols + [f"{c}_idx" for c in cat_cols]
assembler = VectorAssembler(inputCols=assembled_input, outputCol="features_raw", handleInvalid="keep")


vindex = VectorIndexer(inputCol="features_raw", outputCol="features", maxCategories=5000)

prep_pipe = Pipeline(stages=indexers + [assembler, vindex])
prep_model = prep_pipe.fit(train_df)

train_ready = (prep_model.transform(train_df)
               .select("date","target_date","horizon","store_nbr","item_nbr",
                       F.col(label_col).alias("label"), "features")
               .repartition(24)
               .persist(StorageLevel.DISK_ONLY))

val_ready = (prep_model.transform(val_df)
             .select("date","target_date","horizon","store_nbr","item_nbr",
                     F.col(label_col).alias("label"), "features")
             .repartition(24))

test_ready = (prep_model.transform(test_df)
              .select("date","target_date","horizon","store_nbr","item_nbr",
                      F.col(label_col).alias("label"), "features")
              .repartition(24))

print("\nPrepared DataFrames:")
print("train_ready:", train_ready.count())
print("val_ready:",   val_ready.count())
print("test_ready:",  test_ready.count())

# ==========================================================
# 7) Train GBT with lighter params
# ==========================================================
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=25,
    maxDepth=5,
    maxBins=64,
    stepSize=0.08,
    subsamplingRate=0.6,
    minInstancesPerNode=50
)

eval_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

def eval_model(pred_df, name):
    rmse = eval_rmse.evaluate(pred_df)
    mae  = eval_mae.evaluate(pred_df)
    r2   = eval_r2.evaluate(pred_df)
    sample = (pred_df
              .select(F.expm1("label").alias("y_true"),
                      F.expm1("prediction").alias("y_pred"))
              .sample(False, 0.1, 42).limit(200000))
    rmse_sales = (sample
                  .select(F.pow(F.col("y_true")-F.col("y_pred"), 2).alias("se"))
                  .agg(F.sqrt(F.avg("se")).alias("rmse_sales")).first()["rmse_sales"])
    print(f"\n{name}")
    print(f" RMSE(log) = {rmse:.6f} | MAE(log) = {mae:.6f} | R2 = {r2:.4f}")
    print(f" RMSE(sales) [sample] ≈ {rmse_sales:.3f}")
    return {"rmse_log": rmse, "mae_log": mae, "r2": r2, "rmse_sales": rmse_sales}

print("\n--- Training GBTRegressor (NO OOM version) ---")
gbt_model = gbt.fit(train_ready)

pred_val  = gbt_model.transform(val_ready)
pred_test = gbt_model.transform(test_ready)

eval_model(pred_val,  "GBT (VAL)")
eval_model(pred_test, "GBT (TEST)")

print("\nDone.")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/27 22:10:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Paths:
train: /kaggle/input/corporacin-favorita-grocery-sales-forecasting/train.csv
items: /kaggle/input/corporacin-favorita-grocery-sales-forecasting/items.csv
stores: /kaggle/input/corporacin-favorita-grocery-sales-forecasting/stores.csv
transactions: /kaggle/input/corporacin-favorita-grocery-sales-forecasting/transactions.csv
oil: /kaggle/input/corporacin-favorita-grocery-sales-forecasting/oil.csv
holidays: /kaggle/input/corporacin-favorita-grocery-sales-forecasting/holidays_events.csv

--- Phase 1: Selecting subset ---


Selected families: ['GROCERY I', 'BEVERAGES', 'CLEANING']


Selected stores: [45]


Selected items total: 240

--- Phase 2: Loading filtered tables ---


...

train_ready: 739200


val_ready: 62400


test_ready: 16800

--- Training GBTRegressor (NO OOM version) ---


25/12/27 23:06:08 ERROR Instrumentation: java.lang.IllegalArgumentException: requirement failed: DecisionTree requires maxBins (= 64) to be at least as large as the number of values in each categorical feature, but categorical feature 5 has 541 values. Consider removing this and other categorical features with a large number of values, or add more training examples.
	at scala.Predef$.require(Predef.scala:281)
	at org.apache.spark.ml.tree.impl.DecisionTreeMetadata$.buildMetadata(DecisionTreeMetadata.scala:151)
	at org.apache.spark.ml.tree.impl.GradientBoostedTrees$.boost(GradientBoostedTrees.scala:333)
	at org.apache.spark.ml.tree.impl.GradientBoostedTrees$.run(GradientBoostedTrees.scala:56)
	at org.apache.spark.ml.regression.GBTRegressor.$anonfun$train$1(GBTRegressor.scala:190)
	at org.apache.spark.ml.util.Instrumentation$.$anonfun$instrumented$1(Instrumentation.scala:191)
	at scala.util.Try$.apply(Try.scala:213)
	at org.apache.spark.ml.util.Instrumentation$.instrumented(Instrumentatio

IllegalArgumentException: requirement failed: DecisionTree requires maxBins (= 64) to be at least as large as the number of values in each categorical feature, but categorical feature 5 has 541 values. Consider removing this and other categorical features with a large number of values, or add more training examples.

In [2]:


from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.storagelevel import StorageLevel
from pyspark.sql import functions as F

# --- optional cleanup (щоб не тримати старі кеші) ---
for obj in ["train_ready", "val_ready", "test_ready"]:
    if obj in globals():
        try:
            globals()[obj].unpersist()
        except:
            pass
try:
    spark.catalog.clearCache()
except:
    pass

label_col = "label_target"

# Low-card categoricals only (реально категоріальні)
cat_cols = ["family", "store_type", "city", "state"]

# Numeric (включаючи id)
num_cols = [
    "store_nbr", "item_nbr", "horizon",
    "class", "perishable", "cluster",
    "onpromotion_target",
    *[f"lag{lag}_label" for lag in LAGS],
    *[f"roll{rw}_mean_label" for rw in ROLL_WINDOWS],
    *[f"{c}_t" for c in future_cols],
]

# Fill NA
train_df2 = train_df.fillna({c: 0 for c in num_cols}).fillna({c: "UNKNOWN" for c in cat_cols})
val_df2   = val_df.fillna({c: 0 for c in num_cols}).fillna({c: "UNKNOWN" for c in cat_cols})
test_df2  = test_df.fillna({c: 0 for c in num_cols}).fillna({c: "UNKNOWN" for c in cat_cols})

# Index only small categoricals
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]

assembled_input = num_cols + [f"{c}_idx" for c in cat_cols]
assembler = VectorAssembler(inputCols=assembled_input, outputCol="features_raw", handleInvalid="keep")


vindex = VectorIndexer(inputCol="features_raw", outputCol="features", maxCategories=32)

prep_pipe = Pipeline(stages=indexers + [assembler, vindex])
prep_model = prep_pipe.fit(train_df2)

train_ready = (prep_model.transform(train_df2)
               .select("date", "target_date", "horizon", "store_nbr", "item_nbr",
                       F.col(label_col).alias("label"), "features")
               .repartition(24)
               .persist(StorageLevel.DISK_ONLY))
_ = train_ready.count()  # materialize

val_ready = (prep_model.transform(val_df2)
             .select("date", "target_date", "horizon", "store_nbr", "item_nbr",
                     F.col(label_col).alias("label"), "features")
             .repartition(24))

test_ready = (prep_model.transform(test_df2)
              .select("date", "target_date", "horizon", "store_nbr", "item_nbr",
                      F.col(label_col).alias("label"), "features")
              .repartition(24))

print("\nPrepared DataFrames:")
print("train_ready:", train_ready.count())
print("val_ready:",   val_ready.count())
print("test_ready:",  test_ready.count())

# ---------------- Train ----------------
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="label",
    maxIter=25,
    maxDepth=5,
    maxBins=64,  
    stepSize=0.08,
    subsamplingRate=0.6,
    minInstancesPerNode=50
)

eval_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

def eval_model(pred_df, name):
    rmse = eval_rmse.evaluate(pred_df)
    mae  = eval_mae.evaluate(pred_df)
    r2   = eval_r2.evaluate(pred_df)

    sample = (pred_df
              .select(F.expm1("label").alias("y_true"),
                      F.expm1("prediction").alias("y_pred"))
              .sample(False, 0.1, 42).limit(200000))
    rmse_sales = (sample
                  .select(F.pow(F.col("y_true")-F.col("y_pred"), 2).alias("se"))
                  .agg(F.sqrt(F.avg("se")).alias("rmse_sales"))
                  .first()["rmse_sales"])

    print(f"\n{name}")
    print(f" RMSE(log) = {rmse:.6f} | MAE(log) = {mae:.6f} | R2 = {r2:.4f}")
    print(f" RMSE(sales) [sample] ≈ {rmse_sales:.3f}")
    return {"rmse_log": rmse, "mae_log": mae, "r2": r2, "rmse_sales": rmse_sales}

print("\n--- Training GBTRegressor (fixed categories) ---")
gbt_model = gbt.fit(train_ready)

pred_val  = gbt_model.transform(val_ready)
pred_test = gbt_model.transform(test_ready)

eval_model(pred_val,  "GBT (VAL)")
eval_model(pred_test, "GBT (TEST)")
print("\nDone.")



Prepared DataFrames:
train_ready: 739200


val_ready: 62400


test_ready: 16800

--- Training GBTRegressor (fixed categories) ---



GBT (VAL)
 RMSE(log) = 0.550910 | MAE(log) = 0.409206 | R2 = 0.6914
 RMSE(sales) [sample] ≈ 23.386



GBT (TEST)
 RMSE(log) = 0.918669 | MAE(log) = 0.563696 | R2 = 0.3892
 RMSE(sales) [sample] ≈ 20.533

Done.


In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import *
from pyspark.sql import DataFrame
from pyspark.storagelevel import StorageLevel

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# ---------------- USER SETTINGS ----------------
HORIZON_DAYS = 1     # 1 = next-day, 7 = next-week


N_WORKERS = 2       
SHUFFLE_PARTS = 6
DATA_PARTS = 6

# time split by TARGET DATE
TRAIN_CUT = "2017-06-01"   
TEST_CUT  = "2017-08-01"   

START_DATE = "2015-01-01"
END_DATE   = "2017-08-16"

# subset (keeps run fast + avoids OOM)
K_FAMILIES = 5
K_STORES   = 1
TOP_ITEMS_PER_FAMILY = 40

# lags / windows (only past, exclude "today" from rolling to keep it clean)
LAGS = [1,2,3,4,5,6,7,14,28]
ROLL_WINDOWS = [3,7,14,28]

# Kaggle "test" knows promo in advance -> OK; if you want strict real-world, set False
USE_PROMO_TARGET = True

# target encoding smoothing
TE_ALPHA = 20.0

# optional: small time-aware tuning inside TRAIN
DO_TUNING = True
VAL_DAYS = 42

# which models to train
TRAIN_GBT = True
TRAIN_RF  = True

# ------------------------------------------------------------
# Spark (restart)
# ------------------------------------------------------------
try:
    spark.stop()
except:
    pass

spark = (SparkSession.builder
         .master(f"local[{N_WORKERS}]")
         .appName("favorita-forecast-train-test")
         .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTS))
         .config("spark.default.parallelism", str(max(2, N_WORKERS)))
         .config("spark.sql.adaptive.enabled", "true")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.driver.memory", "6g")
         .config("spark.driver.maxResultSize", "1g")
         .config("spark.memory.fraction", "0.6")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark.catalog.clearCache()
spark.sparkContext.setCheckpointDir("/kaggle/working/spark_checkpoints")

H = int(HORIZON_DAYS)
FORECAST_NAME = "Next-day forecast" if H == 1 else ("Next-week forecast" if H == 7 else "Forecast")
print("Spark:", spark.version, "|", FORECAST_NAME)

# ------------------------------------------------------------
# paths
# ------------------------------------------------------------
BASE = "/kaggle/input"

def find_file(filename: str) -> str:
    for d in glob.glob(os.path.join(BASE, "*")):
        cand = os.path.join(d, filename)
        if os.path.exists(cand):
            return cand
    for root, _, files in os.walk(BASE):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f"Could not find {filename} under {BASE}")

train_path  = find_file("train.csv")
items_path  = find_file("items.csv")
stores_path = find_file("stores.csv")
trans_path  = find_file("transactions.csv")
oil_path    = find_file("oil.csv")
hol_path    = find_file("holidays_events.csv")
print("Paths OK")

# ------------------------------------------------------------
# metrics
# ------------------------------------------------------------
eval_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
eval_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
eval_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

def clip_prediction(df):
    return df.withColumn("prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction")))

def eval_log_metrics(df_pred, name):
    df_pred = clip_prediction(df_pred)
    rmse = float(eval_rmse.evaluate(df_pred))
    mae  = float(eval_mae.evaluate(df_pred))
    r2   = float(eval_r2.evaluate(df_pred))
    print(f"{name}: RMSE(log)={rmse:.6f} | MAE(log)={mae:.6f} | R2={r2:.4f}")
    return {"model": name, "rmse": rmse, "mae": mae, "r2": r2}

def add_sales_cols(df_pred):
    df_pred = clip_prediction(df_pred)
    return (df_pred
            .withColumn("actual_sales", F.expm1("label"))
            .withColumn("pred_sales",   F.expm1("prediction"))
            .withColumn("abs_err", F.abs(F.col("pred_sales") - F.col("actual_sales")))
            .withColumn("se", F.pow(F.col("pred_sales") - F.col("actual_sales"), 2))
            .withColumn("ape", F.when(F.col("actual_sales") > 0,
                                      F.col("abs_err") / F.col("actual_sales")).otherwise(F.lit(None)))
           )

def eval_sales_metrics(df_pred, name):
    df_sales = add_sales_cols(df_pred)
    agg = df_sales.agg(
        F.mean("abs_err").alias("mae"),
        F.sqrt(F.mean("se")).alias("rmse"),
        F.sum("abs_err").alias("sae"),
        F.sum("actual_sales").alias("sy"),
    ).first()
    mae = float(agg["mae"] or 0.0)
    rmse = float(agg["rmse"] or 0.0)
    sae = float(agg["sae"] or 0.0)
    sy  = float(agg["sy"]  or 0.0)
    wape = sae / (sy if sy > 0 else 1.0)
    print(f"{name}: MAE(sales)={mae:.4f} | RMSE(sales)={rmse:.4f} | WAPE={wape:.4f}")
    return {"mae_sales": mae, "rmse_sales": rmse, "wape": wape}

# ------------------------------------------------------------
# plotting
# ------------------------------------------------------------
def plot_bar_compare(models, metric="rmse", title=None):
    plt.figure(figsize=(8,4))
    names = [m["model"] for m in models]
    vals = [m[metric] for m in models]
    bars = plt.bar(names, vals)
    plt.title(title or f"{FORECAST_NAME}: Test {metric} (log) — lower is better")
    for b in bars:
        h = b.get_height()
        plt.text(b.get_x() + b.get_width()/2, h, f"{h:.4f}", ha="center", va="bottom", fontsize=10)
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

def plot_scatter_log(df_pred, title, sample_n=4000, frac=0.35, seed=42):
    df_pred = clip_prediction(df_pred)
    s = (df_pred.select("label","prediction")
         .sample(False, frac, seed)
         .limit(sample_n)
         .toPandas())
    if s.empty:
        print("No rows to plot:", title)
        return
    mn = float(min(s["label"].min(), s["prediction"].min()))
    mx = float(max(s["label"].max(), s["prediction"].max()))
    plt.figure(figsize=(6,6))
    plt.scatter(s["label"], s["prediction"], alpha=0.35)
    plt.plot([mn, mx], [mn, mx])
    plt.title(title)
    plt.xlabel("Actual log1p(sales)")
    plt.ylabel("Predicted log1p(sales)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_residual_hist(df_pred, title, frac=0.35, seed=42):
    df_sales = add_sales_cols(df_pred)
    s = (df_sales.select((F.col("pred_sales") - F.col("actual_sales")).alias("res"))
         .sample(False, frac, seed)
         .toPandas())
    if s.empty:
        print("No rows to plot:", title)
        return
    plt.figure(figsize=(9,4))
    plt.hist(s["res"], bins=60)
    plt.title(title)
    plt.xlabel("Residual (pred - actual) in sales units")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

def plot_wape_by_dow(df_pred, title):
    df_sales = add_sales_cols(df_pred)
    pdf = (df_sales.groupBy("dow_t")
           .agg(F.sum("abs_err").alias("sae"), F.sum("actual_sales").alias("sy"))
           .orderBy("dow_t")
           .toPandas())
    if pdf.empty:
        print("No rows to plot:", title)
        return
    pdf["wape"] = pdf["sae"] / pdf["sy"].replace(0, np.nan)
    plt.figure(figsize=(8,4))
    plt.bar(pdf["dow_t"].astype(int), pdf["wape"])
    plt.title(title)
    plt.xlabel("Target weekday (1..7)")
    plt.ylabel("WAPE")
    plt.tight_layout()
    plt.show()

def plot_top_items_wape(df_pred, topk=20, title=None):
    df_sales = add_sales_cols(df_pred)
    pdf = (df_sales.groupBy("item_nbr")
           .agg(F.sum("abs_err").alias("sae"), F.sum("actual_sales").alias("sy"))
           .withColumn("wape", F.col("sae") / F.when(F.col("sy") > 0, F.col("sy")).otherwise(F.lit(1.0)))
           .orderBy(F.desc("sy"))
           .limit(topk)
           .toPandas())
    if pdf.empty:
        print("No rows to plot:", title)
        return
    plt.figure(figsize=(11,4))
    plt.bar(pdf["item_nbr"].astype(int).astype(str), pdf["wape"])
    plt.title(title or f"{FORECAST_NAME}: WAPE for top-{topk} items (by volume)")
    plt.xlabel("item_nbr")
    plt.ylabel("WAPE")
    plt.xticks(rotation=70)
    plt.tight_layout()
    plt.show()

def plot_timeseries_for_item(df_pred, item_nbr, title_prefix):
    df_sales = add_sales_cols(df_pred)
    pdf = (df_sales.filter(F.col("item_nbr") == int(item_nbr))
           .select("target_date","actual_sales","pred_sales")
           .orderBy("target_date")
           .toPandas())
    if pdf.empty:
        print("No rows for item:", item_nbr)
        return
    plt.figure(figsize=(11,4))
    plt.plot(pdf["target_date"], pdf["actual_sales"], label="Actual")
    plt.plot(pdf["target_date"], pdf["pred_sales"], label="Predicted")
    plt.title(f"{title_prefix} — item {int(item_nbr)}")
    plt.xlabel("Date")
    plt.ylabel("Sales")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_total_timeseries(df_pred, title):
    df_sales = add_sales_cols(df_pred)
    pdf = (df_sales.groupBy("target_date")
           .agg(F.sum("actual_sales").alias("actual_total"),
                F.sum("pred_sales").alias("pred_total"))
           .orderBy("target_date")
           .toPandas())
    if pdf.empty:
        print("No rows to plot:", title)
        return
    plt.figure(figsize=(11,4))
    plt.plot(pdf["target_date"], pdf["actual_total"], label="Actual total")
    plt.plot(pdf["target_date"], pdf["pred_total"], label="Predicted total")
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Total sales")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# Schemas
# ------------------------------------------------------------
train_schema = StructType([
    StructField("id", LongType(), True),
    StructField("date", StringType(), True),
    StructField("store_nbr", IntegerType(), True),
    StructField("item_nbr", IntegerType(), True),
    StructField("unit_sales", DoubleType(), True),
    StructField("onpromotion", StringType(), True),
])
items_schema = StructType([
    StructField("item_nbr", IntegerType(), True),
    StructField("family", StringType(), True),
    StructField("class", IntegerType(), True),
    StructField("perishable", IntegerType(), True),
])
stores_schema = StructType([
    StructField("store_nbr", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("type", StringType(), True),
    StructField("cluster", IntegerType(), True),
])
trans_schema = StructType([
    StructField("date", StringType(), True),
    StructField("store_nbr", IntegerType(), True),
    StructField("transactions", IntegerType(), True),
])
oil_schema = StructType([
    StructField("date", StringType(), True),
    StructField("dcoilwtico", DoubleType(), True),
])
hol_schema = StructType([
    StructField("date", StringType(), True),
    StructField("type", StringType(), True),
    StructField("locale", StringType(), True),
    StructField("locale_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("transferred", BooleanType(), True),
])

# ==========================================================
# Phase 1: Select subset (families / store / items)
# ==========================================================
print("\n--- Phase 1: Select subset ---")
items_full  = spark.read.option("header","true").schema(items_schema).csv(items_path)
stores_full = spark.read.option("header","true").schema(stores_schema).csv(stores_path)

train_min = (spark.read.option("header","true").schema(train_schema).csv(train_path)
             .filter(F.col("date") >= START_DATE)
             .select("store_nbr","item_nbr"))

train_with_family = train_min.join(F.broadcast(items_full.select("item_nbr","family")), "item_nbr", "inner")

top_fam = (train_with_family.groupBy("family").count()
           .orderBy(F.desc("count")).limit(K_FAMILIES).collect())
target_families = [r["family"] for r in top_fam]
print("Selected families:", target_families)

train_fam = train_with_family.filter(F.col("family").isin(target_families))

top_store = (train_fam.groupBy("store_nbr").count()
             .orderBy(F.desc("count")).limit(K_STORES).collect())
target_stores = [int(r["store_nbr"]) for r in top_store]
print("Selected stores:", target_stores)

w_items = Window.partitionBy("family").orderBy(F.desc("count"))
item_counts = (train_fam.filter(F.col("store_nbr").isin(target_stores))
               .groupBy("family","item_nbr").count())

top_items_df = (item_counts
                .withColumn("rn", F.row_number().over(w_items))
                .filter(F.col("rn") <= TOP_ITEMS_PER_FAMILY)
                .select("item_nbr").distinct())

target_items = [int(r["item_nbr"]) for r in top_items_df.collect()]
print("Selected items:", len(target_items))

stores = (stores_full.filter(F.col("store_nbr").isin(target_stores))
          .select("store_nbr","city","state",F.col("type").alias("store_type"),"cluster"))

items = (items_full.filter(F.col("family").isin(target_families))
         .filter(F.col("item_nbr").isin(target_items))
         .select("item_nbr","family","class","perishable"))

del train_min, train_with_family, train_fam, item_counts

# ==========================================================
# Phase 2: Build dense grid + features
# ==========================================================
print("\n--- Phase 2: Build grid + features ---")

train_raw = (spark.read.option("header","true").schema(train_schema).csv(train_path)
             .withColumn("date", F.to_date("date"))
             .filter(F.col("date").between(F.to_date(F.lit(START_DATE)), F.to_date(F.lit(END_DATE))))
             .filter(F.col("store_nbr").isin(target_stores)))

train_raw = train_raw.join(F.broadcast(items.select("item_nbr")), "item_nbr", "left_semi")

sales_actual = (train_raw
    .withColumn("onpromotion_bool",
                F.when(F.lower(F.col("onpromotion")) == "true",  F.lit(1))
                 .when(F.lower(F.col("onpromotion")) == "false", F.lit(0))
                 .otherwise(F.lit(0)))
    .withColumn("unit_sales_clipped",
                F.when(F.col("unit_sales") < 0, F.lit(0.0)).otherwise(F.col("unit_sales")))
    .withColumn("label_today", F.log1p(F.col("unit_sales_clipped")))
    .select("date","store_nbr","item_nbr","label_today","onpromotion_bool")
)

date_dim = spark.sql(
    f"SELECT explode(sequence(to_date('{START_DATE}'), to_date('{END_DATE}'), interval 1 day)) as date"
)

# holidays (known in advance)
hol = (spark.read.option("header","true").schema(hol_schema).csv(hol_path)
       .withColumn("date", F.to_date("date"))
       .filter(F.col("date").between(F.to_date(F.lit(START_DATE)), F.to_date(F.lit(END_DATE)))))

if "locale" not in hol.columns and "lacale" in hol.columns:
    hol = hol.withColumnRenamed("lacale", "locale")

# FIX: transferred can be null -> treat null as False, keep those rows
hol_clean = (hol.withColumn("transferred", F.coalesce(F.col("transferred"), F.lit(False)))
             .filter(F.col("transferred") == F.lit(False))
             .drop("transferred"))

h = hol_clean.alias("h")
st = stores.alias("st")

hol_nat = (h.filter(F.col("h.locale") == "National")
           .crossJoin(F.broadcast(stores.select("store_nbr")).alias("s2"))
           .select(F.col("h.date").alias("date"),
                   F.col("s2.store_nbr").alias("store_nbr"),
                   F.col("h.type").alias("type")))

hol_reg = (h.filter(F.col("h.locale") == "Regional")
           .join(F.broadcast(stores.select("store_nbr","state")).alias("s2"),
                 F.col("s2.state") == F.col("h.locale_name"), "inner")
           .select(F.col("h.date").alias("date"),
                   F.col("s2.store_nbr").alias("store_nbr"),
                   F.col("h.type").alias("type")))

hol_loc = (h.filter(F.col("h.locale") == "Local")
           .join(F.broadcast(stores.select("store_nbr","city")).alias("s2"),
                 F.col("s2.city") == F.col("h.locale_name"), "inner")
           .select(F.col("h.date").alias("date"),
                   F.col("s2.store_nbr").alias("store_nbr"),
                   F.col("h.type").alias("type")))

hol_store = hol_nat.unionByName(hol_reg).unionByName(hol_loc)

holiday_features = (hol_store.groupBy("date","store_nbr").agg(
    F.count("*").alias("holiday_count"),
    F.max((F.col("type") == "Holiday").cast("int")).alias("is_holiday"),
    F.max((F.col("type") == "Event").cast("int")).alias("is_event"),
))

# transactions
transactions = (spark.read.option("header","true").schema(trans_schema).csv(trans_path)
                .withColumn("date", F.to_date("date"))
                .filter(F.col("date").between(F.to_date(F.lit(START_DATE)), F.to_date(F.lit(END_DATE))))
                .filter(F.col("store_nbr").isin(target_stores))
                .select("date","store_nbr","transactions"))

# oil forward fill (small)
oil_pd = (spark.read.option("header","true").schema(oil_schema).csv(oil_path)
          .withColumn("date", F.to_date("date"))
          .select("date","dcoilwtico")
          .toPandas())
oil_pd["date"] = pd.to_datetime(oil_pd["date"])
full_dates = pd.date_range(start=START_DATE, end=END_DATE, freq="D")
oil_pd = oil_pd.set_index("date").reindex(full_dates).sort_index()
oil_pd["oil_ffill"] = oil_pd["dcoilwtico"].ffill().fillna(0.0)
oil_pd = oil_pd.reset_index().rename(columns={"index":"date"})[["date","oil_ffill"]]
oil = spark.createDataFrame(oil_pd).withColumn("date", F.to_date("date"))

# deterministic store-date features (safe for future)
pi = float(np.pi)
store_date_det = (date_dim.crossJoin(F.broadcast(stores.select("store_nbr")))
    .withColumn("dow", F.dayofweek("date"))
    .withColumn("dom", F.dayofmonth("date"))
    .withColumn("month", F.month("date"))
    .withColumn("year", F.year("date"))
    .withColumn("weekofyear", F.weekofyear("date"))
    .withColumn("doy", F.dayofyear("date"))
    .withColumn("is_weekend", (F.col("dow").isin([1,7])).cast("int"))
    .withColumn("sin_doy", F.sin(F.lit(2*pi) * F.col("doy") / F.lit(365.0)))
    .withColumn("cos_doy", F.cos(F.lit(2*pi) * F.col("doy") / F.lit(365.0)))
    .withColumn("time_idx", F.datediff(F.col("date"), F.to_date(F.lit(START_DATE))))
    .join(holiday_features, ["date","store_nbr"], "left")
    .fillna({"holiday_count":0,"is_holiday":0,"is_event":0})
)

# store-date numeric signals (use ONLY past, exclude current day)
store_date_num = (store_date_det
    .join(transactions, ["date","store_nbr"], "left")
    .join(oil, ["date"], "left")
    .fillna({"transactions":0, "oil_ffill":0.0})
)

w_store = Window.partitionBy("store_nbr").orderBy("date")

store_date_num = (store_date_num
    .withColumn("transactions_lag1", F.lag("transactions", 1).over(w_store))
    .withColumn("transactions_roll7",  F.avg("transactions").over(w_store.rowsBetween(-7, -1)))
    .withColumn("transactions_roll28", F.avg("transactions").over(w_store.rowsBetween(-28, -1)))
    .withColumn("oil_lag1", F.lag("oil_ffill", 1).over(w_store))
    .withColumn("oil_roll7",  F.avg("oil_ffill").over(w_store.rowsBetween(-7, -1)))
    .withColumn("oil_roll28", F.avg("oil_ffill").over(w_store.rowsBetween(-28, -1)))
    .fillna({
        "transactions_lag1":0, "transactions_roll7":0, "transactions_roll28":0,
        "oil_lag1":0.0, "oil_roll7":0.0, "oil_roll28":0.0
    })
)

# store-item universe
store_item = sales_actual.select("store_nbr","item_nbr").distinct()

# base grid
base_grid = (date_dim.crossJoin(store_item)
    .join(sales_actual, ["date","store_nbr","item_nbr"], "left")
    .fillna({"label_today":0.0,"onpromotion_bool":0})
    .join(store_date_num.select(
        "date","store_nbr",
        "dow","dom","month","year","weekofyear","doy","is_weekend","sin_doy","cos_doy","time_idx",
        "holiday_count","is_holiday","is_event",
        "transactions_lag1","transactions_roll7","transactions_roll28",
        "oil_lag1","oil_roll7","oil_roll28"
    ), ["date","store_nbr"], "left")
    .join(F.broadcast(stores), ["store_nbr"], "left")
    .join(F.broadcast(items), ["item_nbr"], "left")
    .fillna({"family":"UNKNOWN","city":"UNKNOWN","state":"UNKNOWN","store_type":"UNKNOWN"})
    .fillna({"class":0,"perishable":0,"cluster":0})
)

# lags / rolls (past only, exclude "today"; "label_today" remains separate as strongest current signal)
w_si = Window.partitionBy("store_nbr","item_nbr").orderBy("date")

for lag in LAGS:
    base_grid = base_grid.withColumn(f"lag{lag}_label", F.lag("label_today", lag).over(w_si))
    base_grid = base_grid.withColumn(f"lag{lag}_promo",  F.lag("onpromotion_bool", lag).over(w_si))

for rw in ROLL_WINDOWS:
    base_grid = base_grid.withColumn(f"roll{rw}_mean_label", F.avg("label_today").over(w_si.rowsBetween(-rw, -1)))
    base_grid = base_grid.withColumn(f"roll{rw}_mean_promo", F.avg("onpromotion_bool").over(w_si.rowsBetween(-rw, -1)))

# day-of-week history (previous 4 same-dow only)
w_dow = Window.partitionBy("store_nbr","item_nbr","dow").orderBy("date").rowsBetween(-4, -1)
base_grid = base_grid.withColumn("dow_last4_mean_label", F.avg("label_today").over(w_dow))

# trends
base_grid = base_grid.withColumn("trend_7_28", F.col("lag7_label") - F.col("lag28_label"))
base_grid = base_grid.withColumn("trend_1_7",  F.col("lag1_label") - F.col("lag7_label"))

# fill early-history nulls
fill_map = {f"lag{lag}_label":0.0 for lag in LAGS}
fill_map.update({f"lag{lag}_promo":0.0 for lag in LAGS})
fill_map.update({f"roll{rw}_mean_label":0.0 for rw in ROLL_WINDOWS})
fill_map.update({f"roll{rw}_mean_promo":0.0 for rw in ROLL_WINDOWS})
fill_map.update({"dow_last4_mean_label":0.0, "trend_7_28":0.0, "trend_1_7":0.0})
base_grid = base_grid.fillna(fill_map)

# cut lineage
base_grid = base_grid.coalesce(DATA_PARTS).checkpoint(eager=True)
print("base_grid ready")

# ==========================================================
# Phase 3: Build supervised dataset for ONE horizon (split by TARGET DATE)
# ==========================================================
print("\n--- Phase 3: Build supervised dataset ---")

df_base = (base_grid
    .withColumn("target_date", F.date_add(F.col("date"), F.lit(H)))
    .filter(F.col("target_date") <= F.to_date(F.lit(END_DATE)))
)

target_lookup = base_grid.select(
    F.col("date").alias("target_date"),
    "store_nbr","item_nbr",
    F.col("label_today").alias("label_target"),
    F.col("onpromotion_bool").alias("onpromotion_target")
)

df = (df_base
      .join(target_lookup, ["target_date","store_nbr","item_nbr"], "left")
      .fillna({"label_target":0.0, "onpromotion_target":0})
)

# deterministic target-date features (safe to use)
target_det = (store_date_det
    .select(
        F.col("date").alias("target_date"), "store_nbr",
        F.col("dow").alias("dow_t"),
        F.col("dom").alias("dom_t"),
        F.col("month").alias("month_t"),
        F.col("year").alias("year_t"),
        F.col("weekofyear").alias("weekofyear_t"),
        F.col("doy").alias("doy_t"),
        F.col("is_weekend").alias("is_weekend_t"),
        F.col("sin_doy").alias("sin_doy_t"),
        F.col("cos_doy").alias("cos_doy_t"),
        F.col("time_idx").alias("time_idx_t"),
        F.col("holiday_count").alias("holiday_count_t"),
        F.col("is_holiday").alias("is_holiday_t"),
        F.col("is_event").alias("is_event_t")
    )
)

df = (df.join(target_det, ["target_date","store_nbr"], "left")
        .fillna({c:0 for c in target_det.columns if c not in ["target_date","store_nbr"]})
)

train_all = df.filter(F.col("target_date") < F.to_date(F.lit(TRAIN_CUT)))
test_all  = df.filter((F.col("target_date") >= F.to_date(F.lit(TRAIN_CUT))) &
                      (F.col("target_date") <  F.to_date(F.lit(TEST_CUT))))

train_all = train_all.coalesce(DATA_PARTS).persist(StorageLevel.DISK_ONLY)
test_all  = test_all.coalesce(DATA_PARTS).persist(StorageLevel.DISK_ONLY)
n_train = train_all.count()
n_test  = test_all.count()
print(f"rows train/test: {n_train} / {n_test}")

# ==========================================================
# Phase 4: Target encoding (fit on TRAIN only) + assemble features
# ==========================================================
print("\n--- Phase 4: Target encoding + assemble ---")

label_col = "label_target"
promo_feature = "onpromotion_target" if USE_PROMO_TARGET else "onpromotion_bool"

def build_te(df_train: DataFrame, group_cols, te_name: str, alpha: float, global_mean: float):
    stats = (df_train.groupBy(*group_cols)
             .agg(F.count("*").alias("cnt"), F.mean(label_col).alias("mu")))
    te = (stats
          .withColumn(te_name, (F.col("mu")*F.col("cnt") + F.lit(global_mean)*F.lit(alpha)) / (F.col("cnt")+F.lit(alpha)))
          .select(*group_cols, te_name))
    return te

def apply_te(df_any: DataFrame, te_tables: dict, global_mean: float):
    out = df_any
    for name, (keys, te_df) in te_tables.items():
        out = out.join(F.broadcast(te_df), keys, "left")
    fill_map = {}
    for name, (keys, te_df) in te_tables.items():
        fill_map[name] = global_mean
    return out.fillna(fill_map)

def prepare_features(fit_df: DataFrame, other_dfs: dict):
    """
    Fit target encodings on fit_df, apply to fit_df and other_dfs,
    then assemble features (NO raw store_nbr/item_nbr as numeric features).
    Returns: (fit_feat, other_feat_dict)
    """
    global_mean = float(fit_df.agg(F.mean(label_col)).first()[0] or 0.0)

    te_store_item = build_te(fit_df, ["store_nbr","item_nbr"], "te_store_item", TE_ALPHA, global_mean)
    te_item       = build_te(fit_df, ["item_nbr"],             "te_item",       TE_ALPHA, global_mean)
    te_family     = build_te(fit_df, ["family"],               "te_family",     TE_ALPHA, global_mean)
    te_city       = build_te(fit_df, ["city"],                 "te_city",       TE_ALPHA, global_mean)

    te_tables = {
        "te_store_item": (["store_nbr","item_nbr"], te_store_item),
        "te_item":       (["item_nbr"],            te_item),
        "te_family":     (["family"],              te_family),
        "te_city":       (["city"],                te_city),
    }

    fit2 = apply_te(fit_df, te_tables, global_mean)
    other2 = {k: apply_te(v, te_tables, global_mean) for k, v in other_dfs.items()}

    # FEATURE LIST (FIX: removed raw store_nbr/item_nbr)
    num_cols = [
        # item/store metadata (NOT IDs)
        "class","perishable","cluster",

        # strongest "current day" signals (assumes forecast after day close)
        "label_today",
        "onpromotion_bool",

        # target encodings
        "te_store_item","te_item","te_family","te_city",

        # promo at target day (if allowed)
        promo_feature,

        # lags/rolls (past)
        *[f"lag{lag}_label" for lag in LAGS],
        *[f"roll{rw}_mean_label" for rw in ROLL_WINDOWS],
        *[f"lag{lag}_promo" for lag in LAGS],
        *[f"roll{rw}_mean_promo" for rw in ROLL_WINDOWS],
        "dow_last4_mean_label",
        "trend_7_28","trend_1_7",

        # current date calendar + holiday + time index
        "dow","dom","month","year","weekofyear","doy","is_weekend","sin_doy","cos_doy","time_idx",
        "holiday_count","is_holiday","is_event",

        # store-level signals (past only)
        "transactions_lag1","transactions_roll7","transactions_roll28",
        "oil_lag1","oil_roll7","oil_roll28",

        # target date deterministic
        "dow_t","dom_t","month_t","year_t","weekofyear_t","doy_t","is_weekend_t","sin_doy_t","cos_doy_t","time_idx_t",
        "holiday_count_t","is_holiday_t","is_event_t",
    ]

    def cast_and_fill(d: DataFrame):
        d = d.fillna({c:0 for c in num_cols})
        for c in num_cols:
            d = d.withColumn(c, F.col(c).cast("double"))
        return d

    fit2 = cast_and_fill(fit2)
    other2 = {k: cast_and_fill(v) for k, v in other2.items()}

    assembler = VectorAssembler(inputCols=num_cols, outputCol="features", handleInvalid="keep")

    def assemble(d: DataFrame):
        return (assembler.transform(d)
                .select(
                    "date","target_date",
                    "store_nbr","item_nbr","family","dow_t",
                    F.col(label_col).alias("label"),
                    "features"
                )
                .coalesce(DATA_PARTS)
                .persist(StorageLevel.DISK_ONLY))

    fit_feat = assemble(fit2)
    _ = fit_feat.count()
    fit_feat = fit_feat.checkpoint(eager=True)

    other_feat = {}
    for k, v in other2.items():
        tmp = assemble(v)
        _ = tmp.count()
        other_feat[k] = tmp.checkpoint(eager=True)

    return fit_feat, other_feat

# optional tuning split (inside TRAIN only, by target_date)
if DO_TUNING:
    val_start = F.date_sub(F.to_date(F.lit(TRAIN_CUT)), int(VAL_DAYS))
    train_core = train_all.filter(F.col("target_date") < val_start).persist(StorageLevel.DISK_ONLY)
    val_holdout = train_all.filter((F.col("target_date") >= val_start) &
                                   (F.col("target_date") <  F.to_date(F.lit(TRAIN_CUT)))).persist(StorageLevel.DISK_ONLY)
    _ = train_core.count()
    _ = val_holdout.count()
    print(f"Tuning split rows: {train_core.count()} / {val_holdout.count()}")

    tune_train_feat, tune_other = prepare_features(train_core, {"val": val_holdout})
    val_feat = tune_other["val"]
else:
    train_core = None
    val_holdout = None
    tune_train_feat = None
    val_feat = None

# ==========================================================
# Baselines (clean English names)
# ==========================================================
print("\n--- Baseline models ---")

# Naive forecast:
# - for next-week, a stronger naive is "same weekday last week" => lag7_label
# - otherwise use "today" => label_today
naive_pred_col = "lag7_label" if H == 7 else "label_today"

naive_df = (test_all.select(
    "target_date","store_nbr","item_nbr","family","dow_t",
    F.col("label_target").cast("double").alias("label"),
    F.col(naive_pred_col).cast("double").alias("prediction"),
))
m_naive = eval_log_metrics(naive_df.select("label","prediction"), "Naive Forecast")
_ = eval_sales_metrics(naive_df.select("label","prediction"), "Naive Forecast")

ma7_df = (test_all.select(
    "target_date","store_nbr","item_nbr","family","dow_t",
    F.col("label_target").cast("double").alias("label"),
    F.col("roll7_mean_label").cast("double").alias("prediction"),
))
m_ma7 = eval_log_metrics(ma7_df.select("label","prediction"), "Moving Average (7d)")
_ = eval_sales_metrics(ma7_df.select("label","prediction"), "Moving Average (7d)")

results_test = [m_naive, m_ma7]

# ==========================================================
# Tune hyperparameters (small grid) — English labels
# ==========================================================
def tune_gbt(train_feat: DataFrame, val_feat: DataFrame):
    grid = []
    for maxIter in [140, 220]:
        for maxDepth in [5, 7]:
            for subsamplingRate in [0.70, 0.85]:
                for minNode in [20, 40]:
                    grid.append((maxIter, maxDepth, subsamplingRate, minNode))

    best = None
    for maxIter, maxDepth, subsamplingRate, minNode in grid:
        model = GBTRegressor(
            featuresCol="features", labelCol="label",
            maxIter=int(maxIter),
            maxDepth=int(maxDepth),
            stepSize=0.05,
            subsamplingRate=float(subsamplingRate),
            minInstancesPerNode=int(minNode),
            maxBins=32,
            maxMemoryInMB=128,
            cacheNodeIds=False,
            checkpointInterval=5,
            seed=42
        )
        m = model.fit(train_feat)
        pred = m.transform(val_feat).select("label","prediction")
        rmse = float(eval_rmse.evaluate(clip_prediction(pred)))
        if (best is None) or (rmse < best["val_rmse"]):
            best = {"maxIter": maxIter, "maxDepth": maxDepth, "subsamplingRate": subsamplingRate,
                    "minInstancesPerNode": minNode, "val_rmse": rmse}
        print(f"GBT tuning: iter={maxIter} depth={maxDepth} subsample={subsamplingRate} minNode={minNode} -> val_RMSE={rmse:.6f}")
    print("Best GBT params:", best)
    return best

def tune_rf(train_feat: DataFrame, val_feat: DataFrame):
    grid = []
    for numTrees in [120, 180]:
        for maxDepth in [10, 14]:
            for minNode in [20, 40]:
                grid.append((numTrees, maxDepth, minNode))

    best = None
    for numTrees, maxDepth, minNode in grid:
        model = RandomForestRegressor(
            featuresCol="features", labelCol="label",
            numTrees=int(numTrees),
            maxDepth=int(maxDepth),
            minInstancesPerNode=int(minNode),
            featureSubsetStrategy="sqrt",
            subsamplingRate=0.80,
            maxBins=32,
            maxMemoryInMB=128,
            cacheNodeIds=False,
            checkpointInterval=5,
            seed=42
        )
        m = model.fit(train_feat)
        pred = m.transform(val_feat).select("label","prediction")
        rmse = float(eval_rmse.evaluate(clip_prediction(pred)))
        if (best is None) or (rmse < best["val_rmse"]):
            best = {"numTrees": numTrees, "maxDepth": maxDepth, "minInstancesPerNode": minNode, "val_rmse": rmse}
        print(f"Random Forest tuning: trees={numTrees} depth={maxDepth} minNode={minNode} -> val_RMSE={rmse:.6f}")
    print("Best Random Forest params:", best)
    return best

if DO_TUNING:
    best_gbt = tune_gbt(tune_train_feat, val_feat) if TRAIN_GBT else None
    best_rf  = tune_rf(tune_train_feat, val_feat)  if TRAIN_RF  else None

    tune_train_feat.unpersist()
    val_feat.unpersist()
    train_core.unpersist()
    val_holdout.unpersist()
else:
    best_gbt = {"maxIter": 220, "maxDepth": 6, "subsamplingRate": 0.75, "minInstancesPerNode": 30}
    best_rf  = {"numTrees": 160, "maxDepth": 12, "minInstancesPerNode": 30}

# ==========================================================
# Final training (fit on full TRAIN), evaluation on TEST
# ==========================================================
print("\n--- Train models and evaluate on test set ---")

train_feat, other_feat = prepare_features(train_all, {"test": test_all})
test_feat = other_feat["test"]

def train_and_report(model, name: str):
    mdl = model.fit(train_feat)
    pred_te = mdl.transform(test_feat)

    m_te = eval_log_metrics(pred_te.select("label","prediction"), name)
    _ = eval_sales_metrics(pred_te.select("label","prediction"), name)

    plot_scatter_log(pred_te.select("label","prediction"),
                     f"{FORECAST_NAME}: {name} — Actual vs Predicted (log space)")
    plot_residual_hist(pred_te, f"{FORECAST_NAME}: {name} — Residuals (sales units)")
    plot_wape_by_dow(pred_te, f"{FORECAST_NAME}: {name} — WAPE by target weekday")
    plot_top_items_wape(pred_te, topk=20, title=f"{FORECAST_NAME}: {name} — WAPE for top-20 items")
    plot_total_timeseries(pred_te, f"{FORECAST_NAME}: {name} — Total sales (Actual vs Predicted)")

    # show 3 biggest items by volume
    df_sales = add_sales_cols(pred_te)
    top_items = (df_sales.groupBy("item_nbr")
                 .agg(F.sum("actual_sales").alias("sy"))
                 .orderBy(F.desc("sy"))
                 .limit(3)
                 .toPandas()["item_nbr"].tolist())
    for it in top_items:
        plot_timeseries_for_item(pred_te, it, f"{FORECAST_NAME}: {name} (Actual vs Predicted)")

    return m_te

# Gradient-Boosted Trees
if TRAIN_GBT:
    gbt = GBTRegressor(
        featuresCol="features", labelCol="label",
        maxIter=int(best_gbt["maxIter"]),
        maxDepth=int(best_gbt["maxDepth"]),
        stepSize=0.05,
        subsamplingRate=float(best_gbt.get("subsamplingRate", 0.75)),
        minInstancesPerNode=int(best_gbt.get("minInstancesPerNode", 30)),
        maxBins=32,
        maxMemoryInMB=128,
        cacheNodeIds=False,
        checkpointInterval=5,
        seed=42
    )
    m_gbt = train_and_report(gbt, "Gradient-Boosted Trees")
    results_test.append(m_gbt)

# Random Forest
if TRAIN_RF:
    rf = RandomForestRegressor(
        featuresCol="features", labelCol="label",
        numTrees=int(best_rf["numTrees"]),
        maxDepth=int(best_rf["maxDepth"]),
        minInstancesPerNode=int(best_rf["minInstancesPerNode"]),
        featureSubsetStrategy="sqrt",
        subsamplingRate=0.80,
        maxBins=32,
        maxMemoryInMB=128,
        cacheNodeIds=False,
        checkpointInterval=5,
        seed=42
    )
    m_rf = train_and_report(rf, "Random Forest")
    results_test.append(m_rf)

# ==========================================================
# Summary (test)
# ==========================================================
print("\n=== TEST SUMMARY (log-space RMSE) ===")
for r in results_test:
    print(f"{r['model']}: RMSE={r['rmse']:.6f} | R2={r['r2']:.4f}")

plot_bar_compare(results_test, metric="rmse", title=f"{FORECAST_NAME}: Test RMSE(log) comparison")

# cleanup
train_feat.unpersist()
test_feat.unpersist()
train_all.unpersist()
test_all.unpersist()
spark.catalog.clearCache()
print("Done.")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/28 19:34:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.1 | Next-day forecast
Paths OK

--- Phase 1: Select subset ---


Selected families: ['GROCERY I', 'BEVERAGES', 'CLEANING', 'DAIRY', 'PRODUCE']


Selected stores: [45]


Selected items: 200

--- Phase 2: Build grid + features ---


...

GBT tuning: iter=140 depth=5 subsample=0.85 minNode=40 -> val_RMSE=0.453916


GBT tuning: iter=140 depth=7 subsample=0.7 minNode=20 -> val_RMSE=0.453180


GBT tuning: iter=140 depth=7 subsample=0.7 minNode=40 -> val_RMSE=0.465794


GBT tuning: iter=140 depth=7 subsample=0.85 minNode=20 -> val_RMSE=0.450986


GBT tuning: iter=140 depth=7 subsample=0.85 minNode=40 -> val_RMSE=0.467963
GBT tuning: iter=220 depth=5 subsample=0.7 minNode=20 -> val_RMSE=0.456010
